# Paper #60: The Sun's Supergranulation (2018 Update) — Implementation

## 태양의 초립상 (2018 업데이트) — 구현

**Authors**: François Rincon & Michel Rieutord (2018)

**Journal**: Living Reviews in Solar Physics, 15:6

**DOI**: 10.1007/s41116-018-0013-5

---

### Purpose / 목적

**EN**: This notebook reproduces the qualitative features of supergranulation discussed in Rincon & Rieutord (2018): (1) a synthetic supergranulation cellular flow on a solar patch, (2) the power spectrum peak near spherical harmonic degree $\ell \sim 120$, (3) the Schwarzschild convective instability criterion, (4) extraction of horizontal flow from Doppler-like images via cross-correlation / LCT, and (5) time evolution of a single supergranule at the ~1-day timescale.

**KR**: 본 노트북은 Rincon & Rieutord (2018)에서 논의된 초립상의 정성적 특성을 재현합니다: (1) 태양 패치 위의 합성 초립상 셀 유동, (2) 구면조화 차수 $\ell \sim 120$ 근처의 파워 스펙트럼 피크, (3) Schwarzschild 대류 불안정 기준, (4) 교차 상관 / LCT를 통한 Doppler 유사 이미지에서 수평 유동 추출, (5) 단일 초립상의 ~1일 시간 규모 시간 진화.

## 1. Imports and Setup / 임포트와 설정

In [ ]:
"""Imports for supergranulation implementation.

All code comments and docstrings are in English (Google style) per project convention.
"""

import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage
from scipy.fft import fft2, fftfreq

# Reproducibility
np.random.seed(42)

# Matplotlib defaults
plt.rcParams.update({
    'figure.figsize': (9, 6),
    'font.size': 11,
    'axes.grid': True,
    'grid.alpha': 0.3,
})

# Solar constants (SI, unless otherwise noted)
R_SUN = 6.96e8          # Solar radius, m
OMEGA_SUN = 2.87e-6     # Angular rotation, rad/s (27-day rotation)
L_SG_MM = 32.0          # Supergranulation horizontal scale, Mm
V_SG_HORIZ = 300.0      # Typical horizontal rms velocity, m/s
V_SG_VERT = 25.0        # Typical vertical rms velocity, m/s
TAU_SG_DAYS = 1.5       # Typical lifetime, days

print(f"Supergranulation scale: {L_SG_MM} Mm = {L_SG_MM*1e6:.2e} m")
print(f"Horizontal velocity:    {V_SG_HORIZ} m/s")
print(f"Vertical velocity:      {V_SG_VERT} m/s")
print(f"Lifetime:               {TAU_SG_DAYS} days = {TAU_SG_DAYS*24:.1f} hours")
print(f"Velocity anisotropy:    v_h / v_z = {V_SG_HORIZ/V_SG_VERT:.1f}")

## 2. Synthetic Supergranulation Flow Pattern / 합성 초립상 유동 패턴

**EN**: We construct a divergent cellular flow on a 2D patch of the solar surface, mimicking the horizontal component of the supergranulation pattern. Cells are generated as a superposition of Gaussian-divergent sources placed on a jittered hexagonal lattice with ~32 Mm spacing. At each cell center, flow diverges outward with magnitude ~300 m/s; at boundaries, neighboring cells' flows converge (sink regions).

**KR**: 수평 초립상 패턴을 모방하여 태양 표면 2D 패치 위에 발산 셀 유동을 구성합니다. 셀은 ~32 Mm 간격으로 jittered 육각 격자 위에 배치된 Gaussian 발산 소스의 중첩으로 생성됩니다. 각 셀 중심에서 유동이 ~300 m/s 크기로 바깥으로 발산하고, 경계에서는 인접 셀의 유동이 수렴합니다(하강류 영역).

In [ ]:
"""Construct synthetic supergranulation flow field."""

def hexagonal_lattice(L_mm: float, spacing_mm: float, jitter: float = 0.15):
    """Generate a jittered hexagonal lattice of cell centers.

    Args:
        L_mm: Domain side length in megameters.
        spacing_mm: Typical cell spacing in megameters (~32 Mm for SG).
        jitter: Fractional random displacement of lattice points.

    Returns:
        Nx2 array of (x, y) cell-center positions in megameters.
    """
    dx = spacing_mm
    dy = spacing_mm * np.sqrt(3) / 2
    nx = int(L_mm / dx) + 2
    ny = int(L_mm / dy) + 2
    pts = []
    for j in range(ny):
        offset = 0.5 * dx if j % 2 else 0.0
        for i in range(nx):
            x = i * dx + offset
            y = j * dy
            if 0 <= x <= L_mm and 0 <= y <= L_mm:
                dxr = jitter * dx * (np.random.rand() - 0.5)
                dyr = jitter * dy * (np.random.rand() - 0.5)
                pts.append((x + dxr, y + dyr))
    return np.array(pts)


def supergranulation_flow(grid_shape, L_mm: float, spacing_mm: float = 32.0,
                          v_amp: float = 300.0, cell_radius_mm: float = 12.0):
    """Synthetic 2D supergranulation horizontal flow field.

    Flow is a sum of radially divergent Gaussian sources at cell centers.

    Args:
        grid_shape: (Ny, Nx) shape of output grid.
        L_mm: Domain side length in Mm.
        spacing_mm: Typical SG cell spacing.
        v_amp: Peak horizontal velocity magnitude (m/s).
        cell_radius_mm: Gaussian width of a single cell.

    Returns:
        (vx, vy, X, Y) where vx, vy are m/s and X, Y are Mm.
    """
    Ny, Nx = grid_shape
    x = np.linspace(0, L_mm, Nx)
    y = np.linspace(0, L_mm, Ny)
    X, Y = np.meshgrid(x, y)
    centers = hexagonal_lattice(L_mm, spacing_mm)
    vx = np.zeros_like(X)
    vy = np.zeros_like(Y)
    for cx, cy in centers:
        dx = X - cx
        dy = Y - cy
        r2 = dx ** 2 + dy ** 2
        env = np.exp(-r2 / (2 * cell_radius_mm ** 2))
        vx += v_amp * dx * env / cell_radius_mm
        vy += v_amp * dy * env / cell_radius_mm
    return vx, vy, X, Y


# Build a 200 Mm x 200 Mm patch on 512x512 grid
L_domain_mm = 200.0
N_grid = 512
vx, vy, X_mm, Y_mm = supergranulation_flow((N_grid, N_grid), L_domain_mm,
                                            spacing_mm=L_SG_MM, v_amp=V_SG_HORIZ)

v_mag = np.sqrt(vx ** 2 + vy ** 2)
print(f"Domain: {L_domain_mm} Mm x {L_domain_mm} Mm on {N_grid}x{N_grid} grid")
print(f"Pixel size: {L_domain_mm/N_grid*1e3:.1f} km/pixel")
print(f"Max |v_h|:   {v_mag.max():.0f} m/s")
print(f"RMS |v_h|:   {np.sqrt(np.mean(v_mag**2)):.0f} m/s")

In [ ]:
"""Visualize the synthetic supergranulation flow."""

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Panel (a): speed magnitude with cell structure
im0 = axes[0].imshow(v_mag, origin='lower',
                     extent=[0, L_domain_mm, 0, L_domain_mm],
                     cmap='magma', vmin=0, vmax=v_mag.max())
axes[0].set_title('Horizontal speed $|v_h|$ (m/s) / 수평 속력')
axes[0].set_xlabel('x (Mm)')
axes[0].set_ylabel('y (Mm)')
plt.colorbar(im0, ax=axes[0], label='m/s')

# Panel (b): flow vectors on divergence background
dvx_dx = np.gradient(vx, axis=1)
dvy_dy = np.gradient(vy, axis=0)
div = dvx_dx + dvy_dy

im1 = axes[1].imshow(div, origin='lower',
                     extent=[0, L_domain_mm, 0, L_domain_mm],
                     cmap='RdBu_r', vmin=-np.abs(div).max(), vmax=np.abs(div).max())
skip = 24
axes[1].quiver(X_mm[::skip, ::skip], Y_mm[::skip, ::skip],
               vx[::skip, ::skip], vy[::skip, ::skip],
               scale=8e3, color='black', width=0.002)
axes[1].set_title('Horizontal divergence / 수평 발산 (red=up, blue=down)')
axes[1].set_xlabel('x (Mm)')
axes[1].set_ylabel('y (Mm)')
plt.colorbar(im1, ax=axes[1], label=r'$\nabla_h \cdot v_h$ (arb)')

plt.tight_layout()
plt.show()

print('Interpretation: bright (positive divergence) = diverging cell center (upflow),')
print('dark (negative divergence) = downflow at cell boundaries — matches observations.')

## 3. Power Spectrum and the $\ell \sim 120$ Peak / 파워 스펙트럼과 $\ell \sim 120$ 피크

**EN**: Rincon et al. (2017) and earlier work show that the horizontal kinetic energy spectrum peaks at spherical harmonic degree $\ell \approx 120$-140. For a localized Cartesian patch, we work with horizontal wavenumber $k$ and convert via $\ell = k R_\odot$. We compute the azimuthally-averaged 2D power spectrum of the synthetic flow and identify the peak.

**KR**: Rincon et al.(2017)과 이전 연구는 수평 운동 에너지 스펙트럼이 구면조화 차수 $\ell \approx 120$-140에서 피크임을 보여줍니다. 국소 직교 패치의 경우 수평 파수 $k$로 작업하고 $\ell = k R_\odot$으로 변환합니다. 합성 유동의 방위각 평균 2D 파워 스펙트럼을 계산하고 피크를 식별합니다.

In [ ]:
"""Compute azimuthally-averaged 2D power spectrum and convert k -> ell."""

def radial_power_spectrum(field, dx_m: float):
    """Azimuthally-averaged 2D power spectrum.

    Args:
        field: 2D real-valued field on uniform grid.
        dx_m: Pixel size in meters.

    Returns:
        (k_1d, P_1d): 1D wavenumber (1/m) and power spectral density.
    """
    ny, nx = field.shape
    F = fft2(field)
    P = (np.abs(F) ** 2) / (nx * ny)
    kx = fftfreq(nx, d=dx_m) * 2 * np.pi
    ky = fftfreq(ny, d=dx_m) * 2 * np.pi
    KX, KY = np.meshgrid(kx, ky)
    K = np.sqrt(KX ** 2 + KY ** 2)
    dk = 2 * np.pi / (nx * dx_m)
    k_bins = np.arange(0, K.max() + dk, dk)
    k_1d = 0.5 * (k_bins[:-1] + k_bins[1:])
    P_1d = np.zeros_like(k_1d)
    for i in range(len(k_1d)):
        mask = (K >= k_bins[i]) & (K < k_bins[i + 1])
        if mask.any():
            P_1d[i] = P[mask].mean()
    return k_1d, P_1d


dx_m = (L_domain_mm * 1e6) / N_grid
k_arr, Pvx = radial_power_spectrum(vx, dx_m)
_, Pvy = radial_power_spectrum(vy, dx_m)
P_KE = 0.5 * (Pvx + Pvy)

ell_arr = k_arr * R_SUN

valid = (ell_arr > 5) & (ell_arr < 500)
peak_idx = np.argmax(P_KE * valid)
ell_peak = ell_arr[peak_idx]
lam_peak_mm = 2 * np.pi / k_arr[peak_idx] / 1e6

print(f"Spectral peak at ell = {ell_peak:.0f}")
print(f"Corresponding wavelength = {lam_peak_mm:.1f} Mm")
print(f"(Observational value from paper: ell ~ 120-140, wavelength ~30-35 Mm)")

In [ ]:
"""Plot the KE power spectrum vs spherical harmonic degree ell."""

fig, ax = plt.subplots(figsize=(10, 6))
mask = (ell_arr > 5) & (ell_arr < 2000)
ax.loglog(ell_arr[mask], P_KE[mask], 'b-', lw=2, label='Synthetic SG flow spectrum')
ax.axvline(120, color='red', ls='--', lw=1.5, label=r'$\ell=120$ (observed SG peak)')
ax.axvline(140, color='red', ls=':', lw=1.0)
ax.axvline(ell_peak, color='green', ls='--', lw=1.5,
           label=f'Our peak: $\\ell={ell_peak:.0f}$')
ax.set_xlabel(r'Spherical harmonic degree $\ell = k R_\odot$')
ax.set_ylabel(r'Horizontal KE spectral density $E_h(k)$ (arb. units)')
ax.set_title('Power spectrum of synthetic supergranulation / 합성 초립상의 파워 스펙트럼')
ax.legend()
ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.show()

print('Note: our synthetic field was built by design to peak near the 32 Mm scale,')
print('so recovering ell~120 is a sanity check, not independent confirmation.')
print('It demonstrates the scale that observations and theory must reconcile.')

## 4. Schwarzschild Convective Instability Criterion / Schwarzschild 대류 불안정 기준

**EN**: The SCZ is unstable when the actual temperature gradient is steeper than adiabatic: $|dT/dz|_{\text{actual}} > |dT/dz|_{\text{ad}} = g/c_p$. We visualize a simplified solar near-surface profile (using mixing-length / standard model values) to show where convective instability lives.

**KR**: SCZ는 실제 온도 기울기가 단열보다 가파를 때 불안정합니다: $|dT/dz|_{\text{actual}} > |dT/dz|_{\text{ad}} = g/c_p$. 단순화된 태양 근표면 프로파일을 시각화하여 대류 불안정이 존재하는 위치를 보여줍니다.

In [ ]:
"""Schwarzschild criterion: compare actual vs adiabatic T gradients."""

def adiabatic_gradient(g: float, c_p: float) -> float:
    """Return dT/dz for dry adiabat: -g/c_p.

    Args:
        g: Gravitational acceleration, m/s^2.
        c_p: Specific heat at constant pressure, J/(kg K).

    Returns:
        Adiabatic dT/dz, K/m (negative).
    """
    return -g / c_p


# Solar surface values
g_sun = 274.0
c_p_sun = 2.0e4

grad_ad = adiabatic_gradient(g_sun, c_p_sun)
print(f"Adiabatic gradient (dT/dz)_ad = {grad_ad:.3e} K/m  = {grad_ad*1e6:.2f} K/Mm")

# Schematic near-surface profile
z_mm = np.linspace(-30, 2, 500)
z_m = z_mm * 1e6

excess = 3.0e-3 * np.exp(-((z_mm + 0.25) / 0.3) ** 2)
grad_actual = grad_ad - excess
grad_actual = np.where(z_mm > 0, grad_ad * 0.5, grad_actual)

super_adiabatic = grad_actual - grad_ad

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
ax1.plot(grad_actual * 1e6, z_mm, 'b-', lw=2, label='Actual $dT/dz$')
ax1.plot(grad_ad * np.ones_like(z_mm) * 1e6, z_mm, 'r--', lw=2, label='Adiabatic $dT/dz$')
ax1.axhline(0, color='gray', ls=':', label='Photosphere')
ax1.set_xlabel('$dT/dz$ (K/Mm)')
ax1.set_ylabel('Depth z (Mm; 0 = photosphere)')
ax1.set_title('Temperature gradients / 온도 기울기')
ax1.legend()

unstable = super_adiabatic < 0
ax2.plot(super_adiabatic * 1e6, z_mm, 'g-', lw=2)
ax2.fill_betweenx(z_mm, super_adiabatic * 1e6, 0,
                   where=unstable, alpha=0.3, color='orange',
                   label='Unstable (convective)')
ax2.axvline(0, color='k', ls='-', lw=0.7)
ax2.axhline(0, color='gray', ls=':')
ax2.set_xlabel(r'Superadiabatic excess $dT/dz - (dT/dz)_{\rm ad}$ (K/Mm)')
ax2.set_ylabel('Depth z (Mm)')
ax2.set_title('Schwarzschild: unstable where curve < 0')
ax2.legend()

plt.tight_layout()
plt.show()

print('The thin superadiabatic layer just below the photosphere is the thermal boundary')
print('layer driving granulation. The deep SCZ is only weakly superadiabatic — a key fact')
print('for understanding why supergranulation is NOT set by this boundary layer alone.')

## 5. Horizontal Flow from Doppler-like Images via Cross-Correlation / 교차 상관을 통한 수평 유동 추출

**EN**: Local Correlation Tracking (LCT; November & Simon 1988) infers horizontal flow by cross-correlating small sub-images at times $t$ and $t+\Delta t$. We demonstrate the core idea: take two 'snapshots' of a tracer field (here, modeled as an intensity field advected by the SG flow) separated by $\Delta t$, and recover the flow by maximum cross-correlation in local windows.

**KR**: 국소 상관 추적(LCT; November & Simon 1988)은 시간 $t$와 $t+\Delta t$의 작은 하위 이미지를 교차 상관시켜 수평 유동을 추론합니다. 핵심 아이디어를 시연합니다: SG 유동에 의해 이류되는 추적자 장(여기서는 복사강도 장으로 모델)의 두 '스냅샷'을 $\Delta t$ 간격으로 취하고, 국소 윈도우에서 최대 교차 상관으로 유동을 복원합니다.

In [ ]:
"""LCT-style flow recovery from two advected tracer images."""

def advect_tracer(tracer, vx, vy, dt_s: float, dx_m: float):
    """Semi-Lagrangian advection of a tracer field.

    Args:
        tracer: 2D scalar field.
        vx, vy: Velocity components in m/s.
        dt_s: Time step in seconds.
        dx_m: Pixel size in meters.

    Returns:
        Advected tracer field (same shape).
    """
    ny, nx = tracer.shape
    yy, xx = np.meshgrid(np.arange(ny), np.arange(nx), indexing='ij')
    x_dep = xx - (vx * dt_s / dx_m)
    y_dep = yy - (vy * dt_s / dx_m)
    return ndimage.map_coordinates(tracer, [y_dep, x_dep], order=1, mode='reflect')


def lct_flow(img1, img2, box_size: int = 32, dt_s: float = 1.0, dx_m: float = 1.0):
    """Local Correlation Tracking between two images.

    For each box, maximize cross-correlation to find displacement.

    Args:
        img1: 2D image at time t.
        img2: 2D image at time t+dt.
        box_size: Local window size in pixels.
        dt_s: Time separation (s).
        dx_m: Pixel size (m).

    Returns:
        (u, v, Xc, Yc): velocity components (m/s) at box centers.
    """
    ny, nx = img1.shape
    half = box_size // 2
    shift_max = box_size // 4
    centers_y = np.arange(half, ny - half, box_size)
    centers_x = np.arange(half, nx - half, box_size)
    u = np.zeros((len(centers_y), len(centers_x)))
    v = np.zeros_like(u)
    for iy, cy in enumerate(centers_y):
        for ix, cx in enumerate(centers_x):
            w1 = img1[cy - half:cy + half, cx - half:cx + half]
            best_c = -np.inf
            best_dx, best_dy = 0, 0
            for dy in range(-shift_max, shift_max + 1):
                for dx in range(-shift_max, shift_max + 1):
                    y0 = cy + dy - half
                    x0 = cx + dx - half
                    if y0 < 0 or x0 < 0 or y0 + box_size > ny or x0 + box_size > nx:
                        continue
                    w2 = img2[y0:y0 + box_size, x0:x0 + box_size]
                    c = np.corrcoef(w1.ravel(), w2.ravel())[0, 1]
                    if c > best_c:
                        best_c = c
                        best_dx, best_dy = dx, dy
            u[iy, ix] = best_dx * dx_m / dt_s
            v[iy, ix] = best_dy * dx_m / dt_s
    Xc, Yc = np.meshgrid(centers_x, centers_y)
    return u, v, Xc, Yc


# Build a tracer: granule-like noisy intensity field
tracer = np.random.randn(N_grid, N_grid)
tracer = ndimage.gaussian_filter(tracer, sigma=4.0)

dt_min = 30.0
dt_s = dt_min * 60.0
tracer_later = advect_tracer(tracer, vx, vy, dt_s, dx_m)

print(f"Tracer snapshots separated by {dt_min} minutes")
print(f"Expected displacement at v=300 m/s: {300*dt_s/1e6:.2f} Mm = {300*dt_s/dx_m:.1f} pixels")

In [ ]:
"""Run LCT on a subsampled grid for speed and compare to truth."""

ds = 4
tr1 = tracer[::ds, ::ds]
tr2 = tracer_later[::ds, ::ds]
dx_ds = dx_m * ds
vx_true_ds = vx[::ds, ::ds]
vy_true_ds = vy[::ds, ::ds]

u_lct, v_lct, Xc, Yc = lct_flow(tr1, tr2, box_size=16, dt_s=dt_s, dx_m=dx_ds)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
extent = [0, L_domain_mm, 0, L_domain_mm]

speed_true = np.sqrt(vx_true_ds ** 2 + vy_true_ds ** 2)
axes[0].imshow(speed_true, origin='lower', extent=extent, cmap='magma')
sk = 4
axes[0].quiver(np.linspace(0, L_domain_mm, tr1.shape[1])[::sk],
               np.linspace(0, L_domain_mm, tr1.shape[0])[::sk],
               vx_true_ds[::sk, ::sk], vy_true_ds[::sk, ::sk],
               scale=8e3, color='white')
axes[0].set_title('True flow (ground truth) / 실제 유동')
axes[0].set_xlabel('x (Mm)')
axes[0].set_ylabel('y (Mm)')

lct_speed = np.sqrt(u_lct ** 2 + v_lct ** 2)
Xc_mm = Xc * dx_ds / 1e6
Yc_mm = Yc * dx_ds / 1e6
axes[1].imshow(speed_true, origin='lower', extent=extent, cmap='magma', alpha=0.5)
axes[1].quiver(Xc_mm, Yc_mm, u_lct, v_lct, scale=8e3, color='cyan')
axes[1].set_title(f'LCT recovery ({dt_min} min separation) / LCT 복원')
axes[1].set_xlabel('x (Mm)')
axes[1].set_ylabel('y (Mm)')

plt.tight_layout()
plt.show()

print(f"True mean speed:        {speed_true.mean():.0f} m/s")
print(f"LCT-recovered mean:     {lct_speed.mean():.0f} m/s")
print(f"Note: LCT tends to underestimate because granule-scale tracers are noisy below ~2.5 Mm")
print(f"(Rieutord et al. 2010 — the paper's Section 3.1.2 discusses this limit).")

## 6. Time Evolution of a Single Supergranule / 단일 초립상의 시간 진화

**EN**: A single supergranule has a lifetime of ~1-2 days (Parfinenko et al. 2014: 1.3 days; Hirzberger et al. 2008: 1.6-1.8 days). We model this with a simple birth-growth-decay model: a cell grows in amplitude, plateaus, then decays, with characteristic timescale $\tau_{SG} \sim 30$ h.

**KR**: 단일 초립상의 수명은 ~1-2일입니다. 탄생-성장-쇠퇴 모델로 단순 모델링합니다: 셀이 진폭에서 성장하고 평탄화된 후 쇠퇴하며, 특성 시간 규모는 $\tau_{SG} \sim 30$시간입니다.

In [ ]:
"""Model the life cycle amplitude of a single supergranule."""

def lifecycle_amplitude(t_hours: np.ndarray, tau_rise: float = 8.0,
                         tau_decay: float = 20.0, t_peak: float = 12.0,
                         amp_peak: float = 350.0):
    """Birth-growth-decay amplitude model.

    Args:
        t_hours: Array of times in hours since birth.
        tau_rise: Exponential rise timescale (hours).
        tau_decay: Exponential decay timescale (hours).
        t_peak: Time at which plateau begins (hours).
        amp_peak: Peak horizontal velocity amplitude (m/s).

    Returns:
        Array of amplitudes in m/s.
    """
    amp = np.zeros_like(t_hours)
    rising = t_hours < t_peak
    decaying = ~rising
    amp[rising] = amp_peak * (1 - np.exp(-t_hours[rising] / tau_rise))
    amp_at_peak = amp_peak * (1 - np.exp(-t_peak / tau_rise))
    amp[decaying] = amp_at_peak * np.exp(-(t_hours[decaying] - t_peak) / tau_decay)
    return amp


t_hr = np.linspace(0, 60, 400)
amp_t = lifecycle_amplitude(t_hr)

half_max = 0.5 * amp_t.max()
above = t_hr[amp_t > half_max]
fwhm = above.max() - above.min() if len(above) > 0 else np.nan

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(t_hr, amp_t, 'b-', lw=2.5, label='SG horizontal velocity amplitude')
ax.axhline(half_max, color='gray', ls=':', alpha=0.7, label='Half maximum')
ax.axvspan(above.min(), above.max(), alpha=0.15, color='green',
           label=f'FWHM lifetime = {fwhm:.1f} h')
ax.axvline(24, color='red', ls='--', alpha=0.6, label='24 h (1 day)')
ax.axvline(48, color='red', ls='--', alpha=0.6, label='48 h (2 days)')
ax.set_xlabel('Time since cell birth (hours)')
ax.set_ylabel('Horizontal velocity amplitude (m/s)')
ax.set_title('Single supergranule life cycle / 단일 초립상의 생애 주기')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Peak amplitude: {amp_t.max():.0f} m/s  (paper: 300-400 m/s)')
print(f'FWHM lifetime:  {fwhm:.1f} h  (paper: 24-48 h)')

## 7. Key Dimensionless Numbers / 핵심 무차원 수

**EN**: We compute the Rossby number (rotation parameter) and Reynolds number for supergranulation to verify the paper's scaling estimates.

**KR**: 논문의 스케일링 추정치를 확인하기 위해 초립상에 대한 Rossby 수(회전 매개변수)와 Reynolds 수를 계산합니다.

In [ ]:
"""Compute Rossby and Reynolds numbers for supergranulation."""

def rossby_number(v: float, omega: float, L: float) -> float:
    """Rossby number Ro = V / (2 Omega L).

    Args:
        v: Characteristic velocity (m/s).
        omega: Angular rotation rate (rad/s).
        L: Length scale (m).

    Returns:
        Dimensionless Rossby number.
    """
    return v / (2 * omega * L)


def reynolds_number(v: float, L: float, nu: float) -> float:
    """Reynolds number Re = V L / nu.

    Args:
        v: Velocity (m/s).
        L: Length (m).
        nu: Kinematic viscosity (m^2/s).

    Returns:
        Dimensionless Reynolds number.
    """
    return v * L / nu


L_sg_m = L_SG_MM * 1e6
Ro_sg = rossby_number(V_SG_HORIZ, OMEGA_SUN, L_sg_m)

tau_sg_s = TAU_SG_DAYS * 86400
Ro_sg_alt = 1.0 / (2 * OMEGA_SUN * tau_sg_s)

nu_surface = 1e-3
Re_sg = reynolds_number(V_SG_HORIZ, L_sg_m, nu_surface)

ell_nu = L_sg_m * Re_sg ** (-0.75)

Pm = 1e-5
ell_eta = Pm ** (-0.75) * ell_nu

print(f"Supergranulation dimensionless numbers:")
print(f"  Ro (V-based)    = {Ro_sg:.2f}")
print(f"  Ro (tau-based)  = {Ro_sg_alt:.2f}  (paper says ~2-3)")
print(f"  Re              = {Re_sg:.2e}")
print(f"  l_nu (Kolm.)    = {ell_nu:.2e} m = {ell_nu*1000:.2f} mm")
print(f"  l_eta (magn.)   = {ell_eta:.2e} m = {ell_eta:.2f} m")
print()
print("Scale hierarchy comparison / 규모 계층 비교:")
print(f"  l_nu:              {ell_nu:.1e} m")
print(f"  l_eta:             {ell_eta:.1e} m")
print(f"  H_p ~ L_G:         {1e6:.1e} m  (~1 Mm)")
print(f"  L_SG:              {L_sg_m:.1e} m  (~30 Mm)")
print(f"  R_sun:             {R_SUN:.1e} m  (~700 Mm)")

## 8. Summary / 요약

**EN**: This notebook reproduces the key observable and theoretical features of supergranulation discussed in Rincon & Rieutord (2018):

1. **Cellular structure**: 32 Mm horizontal scale, divergent cell centers + convergent boundaries.
2. **Spectral peak** at spherical harmonic degree $\ell \sim 120$ (30-35 Mm wavelength).
3. **Schwarzschild criterion**: SCZ is convectively unstable due to superadiabatic near-surface layer.
4. **LCT methodology**: cross-correlation between time-separated tracer images recovers horizontal flow; resolution limited by granule-scale tracer noise (~2.5 Mm).
5. **Lifetime**: FWHM ~30 hours, consistent with observational 24-48 h.
6. **Dimensionless numbers**: $Ro_{SG} \approx 2$-3 (moderate rotation), $Re \sim 10^{11}$-$10^{13}$ (fully turbulent), $\ell_\nu \sim$ mm scale.

**Open question**: what sets the 30 Mm scale? The 2018 review proposes large-scale near-critical buoyancy-driven convection, but a predictive theory remains elusive. DKIST observations and exascale MHD simulations are expected to advance the field.

**KR**: 본 노트북은 Rincon & Rieutord (2018)에서 논의된 초립상의 주요 관측 및 이론적 특징을 재현합니다:

1. **셀 구조**: 32 Mm 수평 규모, 발산 셀 중심 + 수렴 경계.
2. **스펙트럼 피크**: 구면조화 차수 $\ell \sim 120$(30-35 Mm 파장).
3. **Schwarzschild 기준**: 초단열 근표면 층으로 인해 SCZ는 대류 불안정.
4. **LCT 방법론**: 시간 분리된 추적자 이미지 간 교차 상관이 수평 유동 복원; 입상 규모 추적자 잡음(~2.5 Mm)에 의해 해상도 제한.
5. **수명**: FWHM ~30시간, 관측 24-48시간과 일치.
6. **무차원 수**: $Ro_{SG} \approx 2$-3(중간 정도 회전), $Re \sim 10^{11}$-$10^{13}$(완전 난류), $\ell_\nu \sim$ mm 규모.

**미해결 문제**: 30 Mm 규모를 무엇이 결정하는가? 2018 리뷰는 대규모 임계 근접 부력 구동 대류를 제안하지만 예측 이론은 아직 없습니다. DKIST 관측과 엑사스케일 MHD 시뮬레이션이 분야를 발전시킬 것으로 기대됩니다.